In [2]:
# ==============================================================================
# SCRIPT:         colab_ledger_join_vFinal.ipynb
# OPERATION:      Operation Ledger-Join (Cloud Edition)
# PURPOSE:        Joins the ground-truth asset data (from a CSV in Google Drive)
#                 with the session log (from a live Google Sheet) to produce a
#                 final, reconciled ledger and a discrepancy report.
# ==============================================================================

# --- Step 1: Install Libraries & Import Modules ---
!pip install -q gspread
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default
from google.colab import drive
import os

# --- Step 2: Authenticate and Mount Google Drive ---
print("Authenticating user for Google services...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("✅ G-Sheets access authorized.")

print("\nMounting Google Drive...")
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully at '/content/drive'.")


# --- Step 3: CRITICAL CONFIGURATION ---
# === PLEASE VERIFY THESE PATHS AND NAMES ===

# A. Input file: The ground-truth asset data.
INPUT_ASSET_CSV_PATH = '/content/drive/MyDrive/Opus_1_101001/Database/02_Cleaned_Data/chrono_forge_v6_output.csv'

# B. Input file: The live session log from Google Sheets.
SPREADSHEET_NAME = 'RAW_Requests'
WORKSHEET_NAME = 'raw_requests_polished'

# C. Output file: The destination for the final reconciled ledger.
OUTPUT_ANALYSIS_FOLDER_PATH = '/content/drive/My Drive/Opus_1_101001/Database/03_analysis/'
OUTPUT_FILENAME = 'reconciled_ledger_output.csv'

# D. Column names required for the join operation.
ASSET_FILENAME_COL = 'filename'         # Column name in your asset CSV
LOG_FILENAME_COL = 'Image'              # Column name in your Google Sheet
SESSION_ID_COL_FROM_LOG = 'Session_ID'  # Column name in your Google Sheet for discrepancy check


# --- Step 4: MAIN EXECUTION ---
print("\n--- `Operation Ledger-Join (Cloud Edition)` ---")
try:
    # --- Phase 1: Load Datasets from Cloud Sources ---
    print(f"\nLoading ground-truth asset data...")
    print(f"  > Path: {INPUT_ASSET_CSV_PATH}")
    if not os.path.exists(INPUT_ASSET_CSV_PATH):
        raise FileNotFoundError(f"CRITICAL ERROR: The asset CSV was not found at the specified path.")
    df_assets = pd.read_csv(INPUT_ASSET_CSV_PATH)

    print(f"\nLoading session log data from Google Sheet...")
    print(f"  > Spreadsheet: '{SPREADSHEET_NAME}' | Worksheet: '{WORKSHEET_NAME}'")
    worksheet = gc.open(SPREADSHEET_NAME).worksheet(WORKSHEET_NAME)
    df_log = pd.DataFrame(worksheet.get_all_records())

    print(f"\nFound {len(df_assets)} assets in the asset file.")
    print(f"Found {len(df_log)} entries in the session log.")

    # --- Phase 2: Perform the Left Join ---
    print(f"\nJoining asset data with session log on filename...")
    df_reconciled = pd.merge(
        df_assets,
        df_log,
        left_on=ASSET_FILENAME_COL,
        right_on=LOG_FILENAME_COL,
        how='left'
    )

    # --- Phase 3: Save the Reconciled Ledger to the Analysis Folder ---
    # Ensure the output directory exists before saving
    os.makedirs(OUTPUT_ANALYSIS_FOLDER_PATH, exist_ok=True)
    full_output_path = os.path.join(OUTPUT_ANALYSIS_FOLDER_PATH, OUTPUT_FILENAME)

    df_reconciled.to_csv(full_output_path, index=False)
    print(f"\n✅ Complete reconciled ledger saved to your Google Drive at:")
    print(f"   > '{full_output_path}'")

    # --- Phase 4: Identify and Report Discrepancies ---
    unlogged_assets = df_reconciled[df_reconciled[SESSION_ID_COL_FROM_LOG].isnull()]

    if unlogged_assets.empty:
        print("\n--- Discrepancy Report ---")
        print("✅ No discrepancies found. All assets are accounted for in the session log.")
    else:
        print("\n--- ⚠️ Discrepancy Report: Un-logged Assets Found ⚠️ ---")
        print(f"The following {len(unlogged_assets)} assets exist in your Drive but are MISSING from the session log:")
        report_columns = ['containing_folder', 'filename', 'timestamp']
        print(unlogged_assets[report_columns].to_string(index=False))

except FileNotFoundError as e:
    print(f"\n❌ {e}")
    print("   Please double-check your path configurations in Step 3.")
except Exception as e:
    print(f"\n❌ An unrecoverable error occurred: {e}")

Authenticating user for Google services...
✅ G-Sheets access authorized.

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully at '/content/drive'.

--- `Operation Ledger-Join (Cloud Edition)` ---

Loading ground-truth asset data...
  > Path: /content/drive/MyDrive/Opus_1_101001/Database/02_Cleaned_Data/chrono_forge_v6_output.csv

Loading session log data from Google Sheet...
  > Spreadsheet: 'RAW_Requests' | Worksheet: 'raw_requests_polished'

Found 5273 assets in the asset file.
Found 4775 entries in the session log.

Joining asset data with session log on filename...

✅ Complete reconciled ledger saved to your Google Drive at:
   > '/content/drive/My Drive/Opus_1_101001/Database/03_analysis/reconciled_ledger_output.csv'

--- ⚠️ Discrepancy Report: Un-logged Assets Found ⚠️ ---
The following 289 assets exist in your Drive but are MISSING from the se